# 02 — Data Cleaning

Turns the raw CSVs from notebook 01 into a clean, queryable DuckDB database: load each
source into typed tables, standardise on UTC, melt the wide ENTSO-E generation feed to
long format, and resample the 15-minute series to hourly. Exploratory analysis of these
tables lives in `03_exploratory_analysis.ipynb`.

| Step | What | Output |
|---|---|---|
| Load | CSV → typed tables · UTC · wide→long · 15-min→hourly | clean DuckDB tables |
| Verify | row counts, gaps, quarter counts, per-fuel variability | QA during the build |

**Database:** `data/processed/austria_energy.duckdb`. All timestamps stored as UTC
`TIMESTAMPTZ`; converted to `Europe/Vienna` only at the display layer.

**Prerequisite:** `01_data_collection.ipynb` has been run — raw CSVs present in `data/raw/`.

## Setup

In [ ]:
# imports, paths, house style, DuckDB connection
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import clean
from src.viz import set_house_style, PALETTE, line_profile

set_house_style()

RAW       = PROJECT_ROOT / "data" / "raw"
EXTERNAL  = PROJECT_ROOT / "data" / "external"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

## 1 · Inspect the raw CSVs

In [ ]:
# inspect before schema design
files = [
    "entsoe_generation_AT.csv",
    "entsoe_load_AT.csv",
    "entsoe_prices_AT.csv",
    "weather_vienna.csv",
]

for fname in files:
    path = RAW / fname
    if not path.exists():
        print(f"!! MISSING: {path}\n")
        continue
    size_kb = path.stat().st_size / 1024
    print("=" * 80)
    print(f"  {fname}   ({size_kb:,.0f} KB)")
    print("=" * 80)
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"L{i}: {line.rstrip()[:160]}")
    print()

## 2 · Load OWID, prices & weather

In [ ]:
# Native-hourly + annual sources

# ── 1. OWID — annual, no TZ, CTAS straight from CSV ─────────────────────
print(f"owid:    {clean.build_owid(con, RAW / 'owid_energy_AUT.csv'):,} rows")

# ── 2. Prices — timestamps already carry offsets, parse to UTC ──────────
print(f"prices:  {clean.build_prices(con, RAW / 'entsoe_prices_AT.csv'):,} rows")

# ── 3. Weather ──────────────────────────────────────────────────────────
print(f"weather: {clean.build_weather(con, RAW / 'weather_vienna.csv'):,} rows")

In [ ]:
# sanity check what's in the DB
# The hourly tables share a UTC timestamp span; owid is annual and has no ts_utc, so
# it's checked by its year range instead — MIN/MAX(ts_utc) on it would just return NaT
# (a not-applicable timestamp), which is correct but unhelpful.
display(con.sql("""
SELECT 'prices'  AS tbl, COUNT(*) AS rows,
        MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM prices
UNION ALL
SELECT 'weather', COUNT(*), MIN(ts_utc), MAX(ts_utc) FROM weather
""").df())

display(con.sql("""
SELECT COUNT(*) AS rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM owid_energy_at
""").df())

## 3 · Build generation & demand (melt → resample)

In [ ]:
# Melt wide→long, then resample 15-min → hourly

# ── 4. Generation ───────────────────────────────────────────────────────
print(f"generation: {clean.build_generation(con, RAW / 'entsoe_generation_AT.csv'):,} rows")

# ── 5. Demand ───────────────────────────────────────────────────────────
print(f"demand:     {clean.build_demand(con, RAW / 'entsoe_load_AT.csv'):,} rows")

In [ ]:
# re-run QA on the populated tables
con.sql("""
SELECT 'generation_15min' AS tbl,
       COUNT(*)                                  AS rows,
       COUNT(DISTINCT fuel_type)                 AS n_fuels,
       COUNT(*) FILTER (WHERE mw IS NULL)        AS nulls,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM generation_15min
UNION ALL
SELECT 'demand_15min', COUNT(*), NULL,
       COUNT(*) FILTER (WHERE demand_mw IS NULL),
       MIN(ts_utc), MAX(ts_utc)
FROM demand_15min
""").df()

In [ ]:
# Quarter-count check: should always be 4
display(con.sql("""
SELECT 'generation' AS tbl, n_quarters, COUNT(*) AS n_rows
FROM generation
GROUP BY n_quarters
UNION ALL
SELECT 'demand', n_quarters, COUNT(*)
FROM demand
GROUP BY n_quarters
ORDER BY tbl, n_quarters
""").df())

# What's actually in Austria's generation mix?
display(con.sql("""
SELECT
    fuel_type,
    flow,
    ROUND(AVG(mw))      AS avg_mw,
    ROUND(MAX(mw))      AS peak_mw,
    ROUND(SUM(mw) / 1e6, 2) AS total_twh   -- MW × hours / 1e6 = TWh
FROM generation
GROUP BY fuel_type, flow
ORDER BY avg_mw DESC NULLS LAST
""").df())

In [ ]:
# variability QA
con.sql("""
SELECT fuel_type,
       COUNT(DISTINCT mw)         AS n_unique_values,
       ROUND(STDDEV(mw), 4)       AS std_mw,
       ROUND(MIN(mw), 2)          AS min_mw,
       ROUND(MAX(mw), 2)          AS max_mw
FROM generation
WHERE flow = 'generation'
GROUP BY fuel_type
ORDER BY n_unique_values
""").df()

## 4 · Build GHG emissions

In [ ]:
# from Eurostat env_air_gge

# ── 6. GHG emissions ────────────────────────────────────────────────────
# Raw CSV is WIDE (one column per year) and stuffs the full CRF sector hierarchy into
# two units. build_ghg keeps Mt CO2-eq (MIO_T), the top-level sectors + the two national
# totals, and reshapes LONG via UNPIVOT — DuckDB's SQL twin of pandas .melt().
print(f"ghg_emissions: {clean.build_ghg(con, RAW / 'eurostat_ghg_AT.csv'):,} rows")

## 5 · Build ESR non-ETS emissions

In [ ]:
# inspect the EEA Effort Sharing workbook
# EEA xlsx files often carry title/metadata rows above the real header, and may split
# emissions / AEAs / targets across sheets — so we look before we load.
esr_xlsx = EXTERNAL / "eea_esr_effort_sharing.xlsx"

xls = pd.ExcelFile(esr_xlsx)                       # needs openpyxl
print("Sheets:", xls.sheet_names, "\n")

for sheet in xls.sheet_names:
    raw = pd.read_excel(xls, sheet_name=sheet, header=None)
    print("=" * 80)
    print(f"SHEET: {sheet!r}   shape = {raw.shape}")
    print("=" * 80)
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(raw.head(12).to_string())

    # locate Austria so we can see how the country axis is keyed
    hits = [(int(r), int(c)) for r, c in zip(*raw.apply(
        lambda col: col.astype(str).str.contains("Austria|^AT$", case=False, regex=True, na=False)
    ).to_numpy().nonzero())][:6]
    print("\n  'Austria'/'AT' appears at (row, col):", hits, "\n")

In [ ]:
# ESR non-ETS emissions (EEA Effort-Sharing workbook)
# ── 7. ESR non-ETS emissions ────────────────────────────────────────────
# build_esr reads sheet 'GHG_ESD' (tidy long, already Mt CO2 eq), keeps the Austria
# slice, and stores the ESD_ESR regime flag: ESD = AR4 GWPs (2013-2020),
# ESR = AR5 GWPs (2021-2030) — kept so RQ6 can handle the accounting change at 2021.
print(f"esr_emissions: {clean.build_esr(con, EXTERNAL / 'eea_esr_effort_sharing.xlsx'):,} rows")

## 6 · Confirm the nine tables & close

In [ ]:
con.sql("SHOW TABLES").df()   # expect 9 tables

In [ ]:
# Close the connection
con.close()

## Database built

`data/processed/austria_energy.duckdb` now holds all nine tables.

| Table | Grain | Source |
|---|---|---|
| `generation` | hourly × (fuel_type, flow) | ENTSO-E (resampled from 15-min) |
| `demand` | hourly | ENTSO-E (resampled from 15-min) |
| `prices` | hourly | ENTSO-E day-ahead |
| `weather` | hourly | Open-Meteo / ERA5 (Vienna) |
| `generation_15min`, `demand_15min` | 15-min staging | ENTSO-E (retained for reproducibility) |
| `owid_energy_at` | annual | Our World in Data |
| `ghg_emissions` | annual | Eurostat `env_air_gge` (reshaped wide→long via `UNPIVOT`) |
| `esr_emissions` | annual | EEA Effort Sharing (non-ETS) |

All timestamps stored as UTC `TIMESTAMPTZ`; conversion to `Europe/Vienna` happens only at
the display layer. Resampling: `AVG(mw)` over `time_bucket(INTERVAL 1 HOUR, ts_utc)`.

Notable: hydro dominates Austria's mix; coal phased out April 2020; solar peak-to-average
ratio ~19× drives the duck-curve dynamics explored in notebook 03.